In [13]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [14]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [15]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code
* Respond ONLY the JSON Object, no markdown, no backticks, no explanation.
"""
    messages = []
    add_user_message(messages, prompt)
    text = chat(messages)

    return json.loads(text)

In [16]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent = 2)

In [17]:
print(dataset)

[{'task': 'Write a Python function that validates an AWS S3 bucket name according to AWS naming rules (3-63 characters, lowercase letters, numbers, and hyphens only, must start and end with letter or number)'}, {'task': 'Write a regular expression to match valid AWS IAM role ARNs in the format arn:aws:iam::account-id:role/role-name'}, {'task': 'Write a Python function that extracts the region and service name from an AWS ARN string'}, {'task': 'Write a JSON object representing an AWS Lambda function configuration with environment variables, timeout, memory, and runtime'}, {'task': 'Write a regular expression to validate AWS access key IDs (20 uppercase alphanumeric characters starting with AKIA)'}, {'task': 'Write a Python function that converts an AWS CloudWatch log timestamp to a human-readable format'}, {'task': 'Write a JSON object representing an AWS EC2 security group ingress rule allowing HTTPS traffic from a specific CIDR block'}, {'task': 'Write a Python function that parses a

In [19]:
def run_prompt(test_case):
  """Merges the prompt and test case input, then return the results"""
  prompt = f"""
  Solve the following task:

  {test_case['task']}
  """
  
  messages = []
  add_user_message(messages, prompt)
  output = chat(messages)

  return output

In [20]:
def run_test_case(test_case):
  """Calls run_prompt, then grades the results"""
  output = run_prompt(test_case)
  
  # TODO: Grading
  score = 10

  return {
    "output": output,
    "test_case": test_case,
    "score": score
  }

In [21]:
def run_eval(dataset):
  """Loads the dataset and calls run test case with each case"""
  results = []

  for test_case in dataset:
    result = run_test_case(test_case)
    results.append(result)

  return results

In [23]:
with open("dataset.json", "r") as f:
  dataset = json.load(f)

results = run_eval(dataset)

In [24]:
print(json.dumps(results, indent = 2))

[
  {
    "output": "# AWS S3 Bucket Name Validator\n\n```python\nimport re\n\ndef validate_s3_bucket_name(bucket_name: str) -> tuple[bool, str]:\n    \"\"\"\n    Validates an AWS S3 bucket name according to AWS naming rules.\n    \n    AWS S3 Bucket Naming Rules:\n    - Must be between 3 and 63 characters long\n    - Can only contain lowercase letters, numbers, and hyphens (-)\n    - Must start with a lowercase letter or number\n    - Must end with a lowercase letter or number\n    - Cannot contain consecutive hyphens\n    - Cannot be formatted as an IP address (e.g., 192.168.1.1)\n    \n    Args:\n        bucket_name: The S3 bucket name to validate\n        \n    Returns:\n        A tuple of (is_valid: bool, message: str)\n    \"\"\"\n    \n    # Check if bucket_name is a string\n    if not isinstance(bucket_name, str):\n        return False, \"Bucket name must be a string\"\n    \n    # Check length\n    if len(bucket_name) < 3:\n        return False, \"Bucket name must be at least 